# Lesson 4.1 — Dashboards Overview: Custom Dashboards vs Dash Studio

## Learning Objectives
- Distinguish between Classic Custom Dashboards and Dash Studio
- Navigate to both dashboard types in the Controller UI
- Create a new Dash Studio dashboard
- Understand when to use each dashboard type

**Reference**: [Dashboards and Reports](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/dashboards-and-reports) | [Dash Studio](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/dashboards-and-reports/dash-studio)

## 1. Two Dashboard Systems

AppDynamics currently has two dashboard builders:

| Feature | Classic Custom Dashboards | Dash Studio |
|---------|--------------------------|-------------|
| **Location** | Dashboards & Reports → Dashboards tab | Dashboards & Reports → Dash Studio tab |
| **Editor** | Drag-and-drop widget-based | Modern visual editor with canvas |
| **ADQL support** | Limited | Full (ADQL Query data binding) |
| **Widget types** | 10+ legacy types | Modern chart types with richer options |
| **Variables** | Not supported | Full variable system |
| **Data bindings** | Metric-based | Metric + ADQL + Custom Query |
| **Time Series widget** | Yes | Yes (required for ADQL series()) |
| **War Rooms** | Yes | Yes |
| **Status** | Legacy (still supported) | New default (>=4.5.15.1) |

> **Recommendation**: Use **Dash Studio** for all new dashboards. Classic dashboards are kept for backward compatibility.

This course (Phases 4 and 6) focuses on **Dash Studio** and **Grafana**.

## 2. Navigating to Dash Studio

### Steps to open Dash Studio:
1. Log in to the Controller UI
2. In the left navigation bar, click **Dashboards & Reports**
3. Click the **Dash Studio** tab (second tab, next to "Dashboards")
4. You'll see a list of existing Dash Studio dashboards

### Creating a new dashboard:
1. Click **+ New Dashboard** (or the blue Add button)
2. Give it a name: e.g., "StoreFront Operations"
3. Optionally set a description
4. Click **Create** — the empty canvas opens

### Dashboard canvas elements:
- **Canvas**: The white grid where you place widgets
- **Widget panel**: Click the `+` button to add new widgets
- **Variables button**: Manage dashboard variables (filter icon)
- **Time range selector**: Controls the time window for all widgets
- **Edit / View mode toggle**: Switch between editing and viewing
- **Save button**: Save your changes

## 3. Dash Studio Dashboard Components

A Dash Studio dashboard consists of:

```
Dashboard
├── Variables  ← filter controls shown at top
│   ├── Variable: selectedApp (dropdown)
│   └── Variable: selectedBT  (multi-select)
│
└── Canvas Layout
    ├── Row 1:
    │   ├── [Metric Widget] Overall App Response Time
    │   ├── [Metric Widget] Calls per Minute
    │   └── [Metric Widget] Error Rate
    ├── Row 2:
    │   └── [Time Series Widget] Response Time over Time (ADQL series())
    └── Row 3:
        ├── [Table Widget] Slowest BTs (ADQL)
        └── [Pie Chart] Errors by Country (ADQL)
```

### Widget types in Dash Studio:

| Widget | Best for |
|--------|----------|
| **Time Series** | Response time trends, call rate over time, ADQL series() |
| **Table** | Slowest BTs, error lists, top N queries |
| **Metric** (Stat/Gauge) | Current value of a metric (calls/min, response time) |
| **Pie / Donut Chart** | Distribution (% by BT, by country, by browser) |
| **Bar Chart** | Comparison across categories |
| **Health Status** | Health rule status indicators |
| **Text / Markdown** | Headings, notes, runbook links |

In [ ]:
# Let's design a dashboard structure in code before building it in the UI
# This is a planning tool — helpful for reviewing layout before implementation

dashboard_plan = {
    "name": "StoreFront Operations Dashboard",
    "variables": [
        {"name": "selectedApp", "type": "dropdown", "default": "StoreFront",
         "options": ["StoreFront", "PaymentService", "InventoryAPI"]},
        {"name": "selectedTier", "type": "dropdown", "default": "All",
         "options": ["All", "WebFrontend", "OrderService", "InventoryAPI"]},
    ],
    "rows": [
        {
            "title": "SLO Summary",
            "widgets": [
                {"type": "Metric",      "title": "Avg Response Time",
                 "data": "Overall Application Performance|Average Response Time (ms)"},
                {"type": "Metric",      "title": "Calls per Minute",
                 "data": "Overall Application Performance|Calls per Minute"},
                {"type": "Metric",      "title": "Error Rate",
                 "data": "Overall Application Performance|Errors per Minute"},
            ]
        },
        {
            "title": "Response Time Trends",
            "widgets": [
                {"type": "Time Series",
                 "title": "Transactions by User Experience",
                 "data": "ADQL: SELECT series(eventTimestamp,'1m'),userExperience,count(*) FROM transactions WHERE applicationName='${selectedApp}'"},
            ]
        },
        {
            "title": "Transaction Analysis",
            "widgets": [
                {"type": "Table",
                 "title": "Top 10 Slowest BTs",
                 "data": "ADQL: SELECT transactionName,count(*),avg(responseTime) FROM transactions ORDER BY avg(responseTime) DESC LIMIT 10"},
                {"type": "Pie Chart",
                 "title": "Errors by Country",
                 "data": "ADQL: SELECT country,count(*) FROM transactions WHERE errorOccurred=true"},
            ]
        }
    ]
}

print(f"Dashboard Plan: {dashboard_plan['name']}")
print(f"\nVariables:")
for v in dashboard_plan['variables']:
    print(f"  ${{{v['name']}}} ({v['type']}) — default: {v['default']}")
    print(f"    options: {v['options']}")

print(f"\nLayout:")
for row in dashboard_plan['rows']:
    print(f"  Row: {row['title']}")
    for w in row['widgets']:
        print(f"    [{w['type']:12}] {w['title']}")
        print(f"              Data: {w['data'][:80]}..." if len(w['data']) > 80 else f"              Data: {w['data']}")

In [ ]:
# EXERCISE 4.1: Design a dashboard for a new scenario
#
# Scenario: Your team wants a "SRE Weekly Review" dashboard that shows:
# - Overall application health for the last 7 days
# - Which Business Transactions breached SLA most often
# - Geographic distribution of errors
# - Browser performance comparison
#
# TASK: Fill in the dashboard_plan dict below with the right widgets, data sources,
# and ADQL queries for each widget.

sre_dashboard = {
    "name": "SRE Weekly Review",
    "variables": [
        # TODO: Add relevant variables
    ],
    "rows": [
        {
            "title": "Application Health (7-day)",
            "widgets": [
                # TODO: Add stat widgets for key metrics
            ]
        },
        {
            "title": "SLA Breach Analysis",
            "widgets": [
                # TODO: Add a table showing BTs that were SLOW or VERY_SLOW most often
                # Hint: ADQL on transactions table, grouped by transactionName, filtered by userExperience
            ]
        },
        {
            "title": "Error and Browser Analysis",
            "widgets": [
                # TODO: Add geographic error chart + browser performance chart
            ]
        }
    ]
}

print("Review your dashboard design:")
print(f"  Variables: {len(sre_dashboard['variables'])}")
total_widgets = sum(len(r['widgets']) for r in sre_dashboard['rows'])
print(f"  Total widgets: {total_widgets}")
for r in sre_dashboard['rows']:
    print(f"  Row '{r['title']}': {len(r['widgets'])} widget(s)")

## Summary

- **Classic Custom Dashboards** = legacy, limited ADQL support
- **Dash Studio** = modern, full ADQL, dashboard variables, richer widget types
- A Dash Studio dashboard has: **Variables** (interactive filters) + **Canvas** (widget layout)
- Widget types: Time Series, Table, Metric/Stat, Pie, Bar, Health Status, Text
- Plan your dashboard layout before building it — know your data sources and variables upfront

---

**Next**: [Lesson 4.2 — Dash Studio Layout and Widgets](lesson-4.2-dash-studio-layout-widgets.ipynb)